# Binned phenotype cross-products (grm_class_tool)

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = cfg.params.get("threads", 4)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

In [ ]:
from aoucov.runner import run_stage
require_ready(cfg, "crossproducts")
TOOL, N_SHARDS, NB, SEED = cfg.tool("grm_class_tool"), cfg.param("n_shards"), cfg.param("nblocks"), cfg.param("seed")
shards = [u for u, _ in target_items(cfg, "grm_shards")]
GRM_ID = shards[0].split(".grm.bin.")[0] + ".grm.id"
WORK = cfg.path("crossproducts")
os.makedirs(f"{WORK}/acc", exist_ok=True)

b = cfg.param("bins")
BINS = f"{WORK}/bins.txt"
subprocess.run([sys.executable, os.path.join(REPO, "src/make_bins.py"), "--out", BINS,
                "--fine-width", str(b["fine"]), "--mid-width", str(b["mid"]), "--coarse-width", str(b["coarse"])], check=True)

phenos = target_items(cfg, "phenotypes")
names = [os.path.basename(u)[:-len(".pheno")] for u, _ in phenos]
PHENO_LIST = f"{WORK}/pheno_list.tsv"
pd.DataFrame({"name": names, "path": [u for u, _ in phenos]}).to_csv(PHENO_LIST, sep="\t", header=False, index=False)

## Accumulate: one pass per shard, every phenotype combo

In [ ]:
def accumulate(cfg, item, uri, meta):
    i = item["i"]
    res = subprocess.run([TOOL, "accumulate", "--grm-id", GRM_ID, "--shard", shards[i - 1],
                          "--parallel", str(i), str(N_SHARDS), "--pheno-list", PHENO_LIST,
                          "--bins", BINS, "--nblocks", str(NB), "--seed", str(SEED),
                          "--out-pattern", f"{WORK}/acc/{{name}}.shard{i}.acc.tsv"],
                         capture_output=True, text=True, check=True)
    meta["info"] = [l for l in res.stderr.splitlines() if l.startswith("[INFO]")]
    pd.DataFrame({"name": names, "acc": [f"{WORK}/acc/{n}.shard{i}.acc.tsv" for n in names]}).to_csv(uri, sep="\t", index=False)

run_stage(cfg, "crossproducts", accumulate, jobs=min(N_SHARDS, THREADS), force=FORCE)

## Merge

In [ ]:
def merge(cfg, item, uri, meta):
    name = f"{item['pheno']}__{item['transform']}__{item['covset']}"
    lst = f"{WORK}/acc/{name}.list"
    open(lst, "w").write("".join(f"{WORK}/acc/{name}.shard{i}.acc.tsv\n" for i in range(1, N_SHARDS + 1)))
    subprocess.run([TOOL, "merge", "--acc-list", lst, "--bins", BINS, "--nblocks", str(NB),
                    "--out-prefix", uri[:-len(".full.tsv")]], check=True, capture_output=True)

run_stage(cfg, "merged", merge, force=FORCE)

## Checks

In [ ]:
rep = Report(cfg, "merged")
grm_ids = set(pd.read_csv(GRM_ID, sep="\t", header=None, dtype=str)[1])
rep.note(n_bins=sum(1 for _ in open(BINS)), bins=json.dumps(b), nblocks=NB)
for uri, item in target_items(cfg, "merged"):
    full = pd.read_csv(uri, sep="\t")
    name = os.path.basename(uri)[:-len(".full.tsv")]
    n = len(grm_ids & set(pd.read_csv(f"{cfg.path('phenotypes')}/{name}.pheno", sep=" ", dtype={"IID": str})["IID"]))
    binned = full.loc[full["class"] != "pooled", "full_n"].sum()
    rep.check(f"{name}: all pairs binned", binned >= 0.999 * n * (n - 1) / 2, f"{binned / (n * (n - 1) / 2):.4%}")

In [ ]:
full = pd.read_csv(target_items(cfg, "merged")[0][0], sep="\t")
f = full[(full["class"] != "pooled") & (full["full_n"] > 20)].groupby("bin_index")["full_n"].sum()
mid = np.loadtxt(BINS).mean(1)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.scatter(mid[f.index - 1], f.values, s=6)
ax.set(yscale="log", xlabel="relatedness bin", ylabel="pairs")
rep.fig(fig, "pairs_per_bin", "bins with <= 20 pairs not shown"); plt.show()
rep.save()